# 02 Data Preparation

### Goal of Preparation: Raw to Analysis Ready
Before I begin any deeper exploratory data analysis, I need to clean/curate this dataset a little bit. As it stands, there are far too many variables (50), and I can tell at first glance many that are unnecessary for my goals. Another issue I need to correct is the early quit games that affect the analysis; notably the games that end before 5 floors are completed involving the player either terminating the run or actively griefing the first 1-3 fights. So I want to prune these bad apples before they spoil the analysis. Another issue is that card_selection is a single column containing the floor of the reward as well as all 3 cards and whether they were selected or not. I need to determine a way to make distinct columns for each of these; leaning towards 4 columns: `card_selected`, `card_selected_floor`, `card_not_selected`, `card_not_selected_floor`. Finally, I need to establish a column for `floors_gained`; this identifies how many floors the player made it to after selecting the card. Something simple like `floors_gained = floor_reached - card_selected_floor`.

NOTE: This is not exhaustive list of what may need modified or cleaned, EDA will expose further necessary actions to be done. The goal of this is to have a first usable dataset draft to start any targeted analysis.

In [1]:
import pandas as pd
import json

df_runs = pd.read_parquet('../raw_data/runs_pre_clean.parquet')
print(df_runs.shape)
df_runs.head()

(123436, 50)


,gold_per_floor,floor_reached,playtime,items_purged,score,play_id,local_time,is_ascension_mode,campfire_choices,neow_cost,...,relics_obtained,event_choices,is_beta,boss_relics,items_purged_floors,is_endless,potions_floor_spawned,killed_by,ascension_level,special_seed
0,"[114, 127, 127, 127, 145, 177, 177, 193, 193, ...",50,3610,"[""Strike_R"", ""Strike_R"", ""Defend_R"", ""Defend_R""]",531,251eb1e0-5bfe-4c74-a86d-2925b198cd9c,20201101005734,False,"[{""data"": ""Anger"", ""floor"": 7, ""key"": ""SMITH""}...",TEN_PERCENT_HP_LOSS,...,"[{""floor"": 6, ""key"": ""Ginger""}, {""floor"": 9, ""...","[{""damage_healed"": 0, ""gold_gain"": 0, ""player_...",False,"[{""not_picked"": [""Fusion Hammer"", ""Black Star""...","[12, 24, 30, 39]",False,"[1, 5, 8, 18, 19, 38]",Awakened One,0,NaN
1,"[118, 135, 145, 145, 165, 165, 240, 240, 264, ...",17,468,"[""Undo""]",157,5366608f-19e5-48d5-85d6-3dd822d4bcb0,20201031195733,False,"[{""data"": ""Defragment"", ""floor"": 6, ""key"": ""SM...",NONE,...,"[{""floor"": 9, ""key"": ""Akabeko""}, {""floor"": 14,...","[{""damage_healed"": 0, ""gold_gain"": 0, ""cards_t...",False,[],[11],False,"[1, 5, 14]",NaN,0,NaN
2,"[99, 99]",0,19,[],0,c393b168-681a-461b-ab36-14b12e65789f,20201101005730,False,[],NONE,...,[],[],False,[],[],False,[],NaN,0,NaN
3,"[119, 133, 143, 155, 3, 28, 28, 59, 59, 89, 99...",51,4500,"[""Strike_G"", ""Strike_G""]",1491,eb57ae10-795b-4236-b195-990eda09c558,20201031165728,True,"[{""data"": ""Glass Knife"", ""floor"": 12, ""key"": ""...",NONE,...,"[{""floor"": 6, ""key"": ""Meat on the Bone""}, {""fl...","[{""damage_healed"": 0, ""gold_gain"": 0, ""player_...",False,"[{""not_picked"": [""SacredBark"", ""Runic Dome""], ...","[30, 48]",False,"[1, 2, 11, 13, 16, 21, 23, 29, 33, 35, 39, 45,...",NaN,13,NaN
4,"[113, 124, 143, 157, 157, 71, 106, 106, 175, 1...",16,1209,[],122,776b6c74-f409-42da-80c9-5e6db93fdb58,20201101075731,False,"[{""floor"": 8, ""key"": ""REST""}, {""data"": ""Infern...",NONE,...,"[{""floor"": 7, ""key"": ""WingedGreaves""}, {""floor...","[{""damage_healed"": 0, ""gold_gain"": 0, ""player_...",False,[],[],False,"[1, 3, 4, 12, 14]",The Guardian,0,NaN


### Filter out early-quit / non-standard runs

Drop endless/daily/trial/pre-chosen-seed runs, and drop runs that ended before floor 5 (quit or griefed early fights, per the abandoned-run issue noted in `00_problem_definition`).

In [2]:
FLOOR_REACHED_MIN = 5

df_filtered = df_runs[
    (~df_runs['is_endless'])
    & (~df_runs['is_daily'])
    & (~df_runs['is_trial'])
    & (~df_runs['chose_seed'])
    & (df_runs['floor_reached'] >= FLOOR_REACHED_MIN)
].copy()

print(f"Dropped {len(df_runs) - len(df_filtered)} rows, {len(df_filtered)} remain")

Dropped 16549 rows, 106887 remain


### Filter out non-standard/custom-mode runs

A normal victorious run tops out at floor 57 (the Act 4 Heart kill; Act 3 boss victories land on floor 51). A handful of runs report `floor_reached` up to 107 despite `is_endless`/`is_daily`/`is_trial`/`chose_seed` all being `False`. Inspecting one showed 4 `'BOSS'` nodes in `path_taken` instead of the normal 3; these are non-standard/custom-mode games where the floor numbering doesn't match a standard run.

That same numbering mismatch also shows up in runs whose `floor_reached` looks ordinary: one run has `floor_reached=52` but a `card_choices` entry at floor 87. So on top of capping `floor_reached`, drop any run where a `card_choices` floor exceeds `floor_reached` — that's the direct invariant a normal run can't violate, and together the two catch the runs responsible for the impossible `floors_gained` values (min -35, max 105) seen below.

In [16]:
FLOOR_REACHED_MAX = 57  # Act 4 Heart kill

too_high = df_filtered['floor_reached'] > FLOOR_REACHED_MAX

def max_pick_floor(card_choices_json):
    events = json.loads(card_choices_json)
    floors = [e['floor'] for e in events if 'floor' in e]
    return max(floors, default=-1)

floor_mismatch = df_filtered['card_choices'].apply(max_pick_floor) > df_filtered['floor_reached']

anomalous = too_high | floor_mismatch
print(f"Dropping {anomalous.sum()} non-standard runs, {len(df_filtered) - anomalous.sum()} remain")

df_filtered = df_filtered[~anomalous].copy()

Dropping 0 non-standard runs, 106829 remain


### Split `card_choices` into a per-pick-event table

Each entry in `card_choices` becomes its own row. `card_selected`/`card_selected_floor` are populated when a card was actually taken; `card_not_selected`/`card_not_selected_floor` are populated when the event was a `"SKIP"`. Exactly one pair is non-null per row.

Also attaching confounders at this step: `current_hp`/`max_hp` (indexed from `current_hp_per_floor`/`max_hp_per_floor` at the pick's floor), `relic_count` (relics obtained on or before the pick's floor, from `relics_obtained`), and the run-level `neow_bonus`/`neow_cost`/`player_experience`/`character_chosen`/`ascension_level`.

Note: the per-floor HP arrays sometimes have a shorter array whose *last* entry still lines up with `floor_reached`, but the first entry is a later floor (e.g. `floor_reached=54` with only 5 entries covering floors 50-54). So HP is looked up by anchoring the index to `floor_reached` from the end of the array, not by the raw floor number.

In [12]:
skipped_events = 0

def hp_at_floor(hp_per_floor, floor_reached, f):
    idx = f - floor_reached + len(hp_per_floor) - 1
    return hp_per_floor[idx] if 0 <= idx < len(hp_per_floor) else None

def explode_card_choices(row):
    global skipped_events
    events = json.loads(row['card_choices'])
    current_hp_per_floor = json.loads(row['current_hp_per_floor'])
    max_hp_per_floor = json.loads(row['max_hp_per_floor'])
    relics_obtained = json.loads(row['relics_obtained'])

    records = []
    for pick_num, event in enumerate(events):
        if 'floor' not in event:
            skipped_events += 1
            continue
        f = int(event['floor'])
        is_skip = event['picked'] == 'SKIP'
        records.append({
            'play_id': row['play_id'],
            'pick_num': pick_num,
            'card_selected': None if is_skip else event['picked'],
            'card_selected_floor': None if is_skip else f,
            'card_not_selected': is_skip,
            'card_not_selected_floor': f if is_skip else None,
            'not_picked_options': event['not_picked'],
            'floor_reached': row['floor_reached'],
            'victory': row['victory'],
            'character_chosen': row['character_chosen'],
            'ascension_level': row['ascension_level'],
            'current_hp': hp_at_floor(current_hp_per_floor, row['floor_reached'], f),
            'max_hp': hp_at_floor(max_hp_per_floor, row['floor_reached'], f),
            'relic_count': sum(1 for r in relics_obtained if r['floor'] <= f),
            'neow_bonus': row['neow_bonus'],
            'neow_cost': row['neow_cost'],
            'player_experience': row['player_experience'],
        })
    return records

df_picks = pd.DataFrame(
    record
    for _, row in df_filtered.iterrows()
    for record in explode_card_choices(row)
)
df_picks['card_selected_floor'] = df_picks['card_selected_floor'].astype('Int64')
df_picks['card_not_selected_floor'] = df_picks['card_not_selected_floor'].astype('Int64')

print(f"Skipped {skipped_events} malformed events missing 'floor'")
print(df_picks.shape)
df_picks.head()

Skipped 15 malformed events missing 'floor'
(1351885, 17)


,play_id,pick_num,card_selected,card_selected_floor,card_not_selected,card_not_selected_floor,not_picked_options,floor_reached,victory,character_chosen,ascension_level,current_hp,max_hp,relic_count,neow_bonus,neow_cost,player_experience
0,251eb1e0-5bfe-4c74-a86d-2925b198cd9c,0,Bludgeon,0,False,<NA>,"[Fiend Fire, Corruption]",50,False,IRONCLAD,0,72.0,72.0,0,THREE_RARE_CARDS,TEN_PERCENT_HP_LOSS,3232
1,251eb1e0-5bfe-4c74-a86d-2925b198cd9c,1,Flex,1,False,<NA>,"[Infernal Blade, Thunderclap]",50,False,IRONCLAD,0,72.0,72.0,0,THREE_RARE_CARDS,TEN_PERCENT_HP_LOSS,3232
2,251eb1e0-5bfe-4c74-a86d-2925b198cd9c,2,Anger,2,False,<NA>,"[Flame Barrier, Body Slam]",50,False,IRONCLAD,0,72.0,72.0,0,THREE_RARE_CARDS,TEN_PERCENT_HP_LOSS,3232
3,251eb1e0-5bfe-4c74-a86d-2925b198cd9c,3,Reckless Charge,5,False,<NA>,"[Pommel Strike, Perfected Strike]",50,False,IRONCLAD,0,49.0,72.0,0,THREE_RARE_CARDS,TEN_PERCENT_HP_LOSS,3232
4,251eb1e0-5bfe-4c74-a86d-2925b198cd9c,4,NaN,<NA>,True,6,"[Thunderclap, Disarm, Cleave]",50,False,IRONCLAD,0,49.0,72.0,1,THREE_RARE_CARDS,TEN_PERCENT_HP_LOSS,3232


### `floors_gained`

Anchored to whichever floor is populated for the row (`card_selected_floor` for picks, `card_not_selected_floor` for skips), so it's defined for both event types.

In [13]:
event_floor = df_picks['card_selected_floor'].fillna(df_picks['card_not_selected_floor'])
df_picks['floors_gained'] = df_picks['floor_reached'] - event_floor

df_picks[['card_selected', 'card_selected_floor', 'card_not_selected', 'card_not_selected_floor', 'floors_gained']].head(10)

,card_selected,card_selected_floor,card_not_selected,card_not_selected_floor,floors_gained
0,Bludgeon,0,False,<NA>,50
1,Flex,1,False,<NA>,49
2,Anger,2,False,<NA>,48
3,Reckless Charge,5,False,<NA>,45
4,NaN,<NA>,True,6,44
5,Flex,8,False,<NA>,42
6,Anger+1,10,False,<NA>,40
7,Battle Trance,11,False,<NA>,39
8,Power Through,14,False,<NA>,36
9,NaN,<NA>,True,18,32


### Check for Mutual Exclusivity

Confirm the selected/not-selected columns are mutually exclusive per row, and check the skip rate and `floors_gained` distribution.

In [14]:
both_null = df_picks['card_selected'].isna() & df_picks['card_selected_floor'].isna() & ~df_picks['card_not_selected']
both_filled = df_picks['card_selected'].notna() & df_picks['card_not_selected']
print("Rows with neither selected nor not-selected populated:", both_null.sum())
print("Rows with both selected and not-selected populated:", both_filled.sum())

print("\nSkip rate:", df_picks['card_not_selected'].mean())
print("\nfloors_gained describe:")
print(df_picks['floors_gained'].describe())

Rows with neither selected nor not-selected populated: 0
Rows with both selected and not-selected populated: 0

Skip rate: 0.2112664908627583

floors_gained describe:
count    1351885.0
mean     18.459879
std      13.297023
min            0.0
25%            8.0
50%           15.0
75%           27.0
max           57.0
Name: floors_gained, dtype: Float64


### Save prepared dataset

In [15]:
df_save = df_picks.copy()
df_save['not_picked_options'] = df_save['not_picked_options'].apply(json.dumps)

df_save.to_parquet('../raw_data/runs_post_prep.parquet', index=False)
print("Saved.")

Saved.
